# Demo Video — FSCG Presentation

Renders a polished 1.CFL clip with pitch-line homography overlay, team-coloured
tracked players, jersey names + numbers, live per-player speed, ball tracking,
and a minimap with the live camera viewport. Designed as a pitch artefact for
the FSCG presentation, not a fully unsupervised pipeline — it leans on a
small amount of manual labelling per clip.

## Workflow

1. **Configure** — pick the match, start timestamp, clip length, and player names.
2. **Render** — produce the demo MP4 with the current pipeline.
3. **Label rendered frames** — scrub through the rendered MP4 and tag
   bad-homography frames.
4. **Calibrate** — for the worst frames, edit the projected pitch lines to
   lock in a ground-truth homography. Saved seeds feed back into the next
   render's smoothed seed track.
5. **Compare** — render pipeline-vs-GT overlays and per-frame pixel error.
6. **Label ball positions** — click the ball every Nth frame so the renderer
   has a continuous ground-truthed ball track instead of relying on YOLO.

Steps 3-6 form an iteration loop: more labels → smoother homography track and
ball overlay → re-render step 2.

In [1]:
import sys, importlib, cv2
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Force reloads — the kernel caches modules across edits, which silently
# masks src/* changes until the next restart otherwise.
import src.run_pnlcalib_video as R; importlib.reload(R)
import src.run_demo as D; importlib.reload(D)

from src.config import Config
from src.camera_motion import CameraMotionTracker
from src.manual_calibration import load_match_seeds
from src.run_pnlcalib_video import (
    load_models, predict_one_frame, project_lines,
    ProjectionSmoother, refine_projection_to_lines, REFINE_STATS,
    check_projection_sanity, check_line_alignment,
)


## 1 · Configuration

Pick the match, where in the video to start, and how long the clip should be.
Pre-roll runs the tracker silently before the rendered window so player IDs and
speed/distance are warm by frame 1.

In [2]:
MATCH               = 'sut-mla'
START_TS            = '1:04:48'   # absolute video timestamp (HH:MM:SS)
DURATION_SEC        = 18
PREROLL_SEC         = 15          # tracker warm-up before START_TS
DEVICE              = 'cuda:0'

BALL_CONF_THRESHOLD = 0.08        # below YOLO default — catch more ball detections
BALL_MAX_GAP        = 30          # frames to extrapolate a missed ball (~1.2s @ 25fps)


## 2 · Load models — run once per session

In [3]:
yolo       = YOLO(Config.resolve_yolo_model())
pnl_models = load_models(DEVICE)
print('Models ready.')


Models ready.


## 3 · Player names + ID remaps

`PLAYER_NAMES` maps a BoT-SORT track ID to `(display_name, team_id, jersey_no)`.
Multiple track IDs can map to the same player — happens when a player exits the
frame and re-enters with a fresh ID.

- `team_id`: 0 = Sutjeska (blue) · 1 = Mladost (yellow) · 2 = Other (ref/staff)
- `jersey_no`: integer, or `None` to omit

`TRACK_REMAPS` patches in mid-clip ID swaps — when two players cross and BoT-SORT
swaps their IDs, list the swap as `(seconds_into_render, {old_id: new_id, ...})`.

In [4]:
PLAYER_NAMES: dict[int, tuple[str, int, int | None]] = {
    # ── SUTJESKA ─────────────────────────────────────────────────────────────
    # Defence
    289: ('A.Babic',       0,  72),  513: ('A.Babic',       0,  72),
    319: ('B. Kopitovic',  0,  15),  519: ('B. Kopitovic',  0,  15),
    106: ('A. Golubovic',  0,   2),  611: ('A. Golubovic',  0,   2),
    339: ('M. Vracar',     0,   4),  653: ('M. Vracar',     0,   4),
    # Midfield
    397: ('D. Hocko',      0,  44),
    382: ('J. Cadjenovic', 0,  20),
    596: ('V. Kalezic',    0,  70),  419: ('V. Kalezic',    0,  70),
    # Attack
    415: ('I. Vukcevic',   0,   7),  656: ('I. Vukcevic',   0,   7),
    535: ('M. Jukovic',    0,  88),  395: ('M. Jukovic',    0,  88),
    428: ('V. Cavor',      0,  24),  621: ('V. Cavor',      0,  24),

    # ── MLADOST ──────────────────────────────────────────────────────────────
    # Goalkeeper
    675: ('B.Radanovic',   1,  31),  716: ('B.Radanovic',   1,  31),
    # Defence
    439: ('M.Badnjar',     1,   3),  651: ('M.Badnjar',     1,   3),
    638: ('Z.Ceklic',      1,   6),  437: ('Z.Ceklic',      1,   6),
    429: ('J.Vujisic',     1,   5),  605: ('J.Vujisic',     1,   5),
    412: ('V.Vickovic',    1,  18),  544: ('V.Vickovic',    1,  18),
    # Midfield
    394: ('D.Kontic',      1,   8),
    392: ('N.Radusinovic', 1,  16),
    389: ('D.Vukovic',     1,  77),
    # Attack
    266: ('J.Roganovic',   1,  29),
    253: ('N.A.Cordoba',   1,  28),
    230: ('L.Knezevic',    1,   7),  604: ('L.Knezevic',    1,   7),

    # ── REFEREES ─────────────────────────────────────────────────────────────
    337: ('Sudija', 2, None),  393: ('Sudija', 2, None),
    502: ('Sudija', 2, None),  494: ('Sudija', 2, None),
}

TRACK_REMAPS: list[tuple[float, dict[int, int]]] = [
    # At second 14 of the rendered window: Cordoba and Vracar's IDs swap.
    # raw 253 → display as Cordoba (339);  raw 653 → display as Vracar (253).
    (14, {253: 339, 653: 253}),
]


## 4 · Iterate on the homography

Three tools to find and fix bad frames in the rendered MP4:

1. **Label** — scrub the rendered MP4 and tag each frame Good / Drift /
   Wrong scale / Wrong overlay / No overlay / Skip.
2. **Calibrate** — for the worst frames, click pitch landmarks on the source
   frame to lock in a ground-truth homography. Saved calibrations are also
   picked up by the next render as runtime seeds.
3. **Compare** — for every calibrated frame, the current pipeline output is
   rendered alongside the ground truth so we can see and quantify deviation.

### 4a · Label rendered frames

Steps through the rendered demo MP4. Click a label button to tag the current
frame and auto-advance by `step` frames. Labels persist live to
`output/demo/diag/frame_labels.json`. Bad-tagged frames are exported as JPGs to
`output/demo/diag/labeled/` for diagnostic review.

In [5]:
import ipywidgets as W
from IPython.display import display
import cv2, json
from pathlib import Path

ts_safe_lbl = START_TS.replace(':', '-')
DEMO_PATH   = Config.OUTPUT_DIR / 'demo' / f'{MATCH}_{ts_safe_lbl}_demo.mp4'
LABELS_PATH = Path('../output/demo/diag/frame_labels.json')
BAD_DIR     = Path('../output/demo/diag/labeled')
LABELS_PATH.parent.mkdir(parents=True, exist_ok=True)
BAD_DIR.mkdir(parents=True, exist_ok=True)

cap_lbl = cv2.VideoCapture(str(DEMO_PATH))
fps_lbl = cap_lbl.get(cv2.CAP_PROP_FPS) or 25.0
nf_lbl  = int(cap_lbl.get(cv2.CAP_PROP_FRAME_COUNT))

labels = ({int(k): v for k, v in json.load(open(LABELS_PATH)).items()}
          if LABELS_PATH.exists() else {})
state = {'idx': 0, 'step': 10}


def _save_labels():
    json.dump({str(k): v for k, v in sorted(labels.items())},
              open(LABELS_PATH, 'w'), indent=1)


def show():
    cap_lbl.set(cv2.CAP_PROP_POS_FRAMES, state['idx'])
    ok, frm = cap_lbl.read()
    if not ok:
        return
    _, buf = cv2.imencode('.jpg', frm, [cv2.IMWRITE_JPEG_QUALITY, 80])
    img_w.value = buf.tobytes()
    counts = {}
    for v in labels.values():
        counts[v] = counts.get(v, 0) + 1
    summary = ' | '.join(f'{k}:{v}' for k, v in sorted(counts.items()))
    status_w.value = (
        "<div style='font-family:monospace'>"
        f"<b>Frame {state['idx']}/{nf_lbl-1}</b> &nbsp; "
        f"t={state['idx']/fps_lbl:.2f}s &nbsp; "
        f"label: <b>{labels.get(state['idx'], '-')}</b><br>"
        f"Labeled: {len(labels)}/{nf_lbl} &nbsp; {summary}"
        "</div>"
    )


def jump(delta):
    state['idx'] = max(0, min(nf_lbl - 1, state['idx'] + delta))
    show()


def tag(label_name, save_bad=False):
    labels[state['idx']] = label_name
    _save_labels()
    if save_bad:
        cap_lbl.set(cv2.CAP_PROP_POS_FRAMES, state['idx'])
        ok, frm = cap_lbl.read()
        if ok:
            cv2.imwrite(str(BAD_DIR / f"f{state['idx']:04d}_{label_name}.jpg"), frm)
    state['idx'] = min(nf_lbl - 1, state['idx'] + state['step'])
    show()


def btn(text, fn):
    b = W.Button(description=text, layout=W.Layout(width='auto'))
    b.on_click(lambda _: fn())
    return b


img_w    = W.Image(format='jpeg', width=1280)
status_w = W.HTML()

step_box = W.IntText(value=state['step'], description='step:',
                     layout=W.Layout(width='130px'))
step_box.observe(lambda c: state.update(step=max(1, int(c['new']))), names='value')

nav = W.HBox([
    btn('<<<<', lambda: jump(-state['step'] * 5)),
    btn('<<',   lambda: jump(-state['step'])),
    btn('<',    lambda: jump(-1)),
    btn('>',    lambda: jump(1)),
    btn('>>',   lambda: jump(state['step'])),
    btn('>>>>', lambda: jump(state['step'] * 5)),
    step_box,
])

labelers = W.HBox([
    btn('✓ Good',          lambda: tag('good')),
    btn('Drift',           lambda: tag('drift', save_bad=True)),
    btn('Wrong scale',     lambda: tag('wrong_scale', save_bad=True)),
    btn('Wrong overlay',   lambda: tag('wrong_overlay', save_bad=True)),
    btn('No overlay',      lambda: tag('no_overlay')),
    btn('Skip',            lambda: tag('skip')),
])

display(W.VBox([status_w, img_w, nav, labelers]))
show()


Run this when you're done labeling for a histogram + bad-frame re-export.

In [6]:
from collections import Counter

labels = {int(k): v for k, v in json.load(open(LABELS_PATH)).items()}
total  = len(labels)

print(f'Total labeled: {total}')
for k, v in Counter(labels.values()).most_common():
    print(f'  {k:15s}: {v:4d} ({100*v/max(total,1):5.1f}%)')

cap = cv2.VideoCapture(str(DEMO_PATH))
bad = sorted(idx for idx, lbl in labels.items()
             if lbl in ('drift', 'wrong_scale', 'wrong_overlay'))
for idx in bad:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frm = cap.read()
    if ok:
        cv2.imwrite(str(BAD_DIR / f'f{idx:04d}_{labels[idx]}.jpg'), frm)
cap.release()
print(f'\nExported {len(bad)} bad frames to {BAD_DIR}')


Total labeled: 62
  drift          :   34 ( 54.8%)
  wrong_overlay  :   16 ( 25.8%)
  no_overlay     :    6 (  9.7%)
  wrong_scale    :    5 (  8.1%)
  skip           :    1 (  1.6%)

Exported 55 bad frames to ..\output\demo\diag\labeled


### 4b · Calibrate ground-truth frames (line-adjust widget)

Steps through every labeled frame from cell 4a (in order), runs the current
pipeline on the corresponding source frame, and pre-projects every named pitch
line as an editable overlay. Use the **◀ / ▶** buttons to move between frames
without re-running the cell. **Bad-only** narrows navigation to frames you
tagged drift / wrong overlay / wrong scale.

**Mental model:** every projected line is *already in the fit* at PnLCalib's
position — including lines whose visible portion misses the frame. Off-frame
lines silently anchor the fit's off-halfway dimensions to PnLCalib's prior, so
even on wide-center shots where only halfway + touchlines are visible the
solver isn't degenerate. Your job is to **fix the visible lines that PnLCalib
got wrong**, not to confirm every line from scratch.

**How to use:**

1. **Drag an endpoint handle** of any solid-coloured line to nudge it onto the
   painted pitch line. Endpoints are auto-clipped to the visible frame so
   they're always grabbable, even when the world endpoint projects off-screen.
   Only the line through the two endpoints matters — drop each handle wherever
   the painted line is clearly visible.
2. Tap a **named arc/spot point** (dropdown) on its painted location to add a
   point correspondence. Click an existing point to remove it. Useful on close
   shots where most lines are off-frame.
3. **Right-click** any solid-coloured line to exclude it from the fit
   (e.g. PnLCalib clearly placed it on a stand or running track). It turns
   gray-dashed; left-click to put it back in the fit.
4. **Re-include all** restores any excluded lines to the fit.
   **Reset to PnLCalib** discards all your edits and re-projects from the
   pipeline's current P.
5. The **cyan dashed** lines are the live re-projection of all named pitch
   lines under the current fitted H — a quality indicator. They should converge
   onto painted lines as you fix the bad ones.
6. **Save** writes `data/manual_calibration/{slug}_frame_{N:07d}.json` —
   picked up by the next demo render and the comparison cell. Then **Next ▶**.

If the pipeline can't produce any P on a frame (close-up, total fail), the
navigator falls back to `build_line_labeling_widget` so you can draw lines
from scratch.

In [7]:
%matplotlib widget
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Reload so source edits to the widget pick up without a kernel restart.
import src.manual_calibration as MC; importlib.reload(MC)
from src.manual_calibration import (
    build_line_adjust_widget, build_line_labeling_widget,
    calibration_path, load_calibration,
)

BAD_LABELS = {'drift', 'wrong_overlay', 'wrong_scale'}

# All labeled demo-frame indices, sorted. Cell 4a tags every Nth frame (default
# step=10) so this list IS your "every 10th frame" iterator.
labels_now = ({int(k): v for k, v in json.load(open(LABELS_PATH)).items()}
               if LABELS_PATH.exists() else {})
target_idxs = sorted(labels_now.keys())
if not target_idxs:
    raise RuntimeError('No labeled frames yet — run cell 4a first.')

cap_src = cv2.VideoCapture(str(Config.MATCH_VIDEOS[MATCH]))
fps_src = cap_src.get(cv2.CAP_PROP_FPS) or 25.0
START_FRAME = int(D.parse_ts(START_TS) * fps_src)

nav = {'pos': 0, 'bad_only': False}


def _filtered_idxs():
    if nav['bad_only']:
        return [i for i in target_idxs if labels_now.get(i) in BAD_LABELS]
    return target_idxs


def render_for(demo_idx: int):
    plt.close('all')   # tear down prior figure so canvases don't pile up
    abs_frame = START_FRAME + demo_idx
    cap_src.set(cv2.CAP_PROP_POS_FRAMES, abs_frame)
    ok, src_frame = cap_src.read()
    if not ok:
        with out:
            clear_output(wait=True)
            print(f'Could not read source frame {abs_frame}')
        return

    fh_src, fw_src = src_frame.shape[:2]
    P_init, init_status = predict_one_frame(src_frame, pnl_models, fw_src, fh_src)
    if P_init is not None:
        P_init = refine_projection_to_lines(P_init, src_frame)

    existing_path = calibration_path(MATCH, abs_frame)
    existing = load_calibration(existing_path) if existing_path.exists() else None

    pool = _filtered_idxs()
    pos_in_pool = pool.index(demo_idx) if demo_idx in pool else -1
    info = (
        f"demo idx {demo_idx}  ({pos_in_pool + 1}/{len(pool)} in pool, "
        f"{nav['pos'] + 1}/{len(target_idxs)} overall)   "
        f"src frame {abs_frame}   label: {labels_now.get(demo_idx, '-')}   "
        f"P: {'OK' if P_init is not None else 'FAIL'} ({init_status})   "
        f"existing GT: {'YES' if existing else 'no'}"
    )

    with out:
        clear_output(wait=True)
        print(info)
        if P_init is not None:
            display(build_line_adjust_widget(
                MATCH, src_frame, abs_frame, P_init, existing=existing,
            ))
        else:
            print('Pipeline P unavailable — falling back to from-scratch widget.')
            display(build_line_labeling_widget(
                MATCH, src_frame, abs_frame, existing=existing,
            ))


def _step(delta):
    if nav['bad_only']:
        # Navigate within the bad-only pool but keep nav['pos'] aligned to the
        # absolute target_idxs index, so toggling the filter off resumes here.
        pool = _filtered_idxs()
        if not pool:
            return
        cur = target_idxs[nav['pos']]
        if cur in pool:
            i = pool.index(cur)
        else:
            # Find the nearest bad frame in the requested direction.
            i = next((j for j, v in enumerate(pool) if v >= cur), len(pool) - 1)
            i = i if delta > 0 else max(0, i - 1)
        i = max(0, min(len(pool) - 1, i + delta))
        nav['pos'] = target_idxs.index(pool[i])
    else:
        nav['pos'] = max(0, min(len(target_idxs) - 1, nav['pos'] + delta))
    render_for(target_idxs[nav['pos']])


def _toggle_bad(_b):
    nav['bad_only'] = not nav['bad_only']
    btn_bad.description = ('All frames' if nav['bad_only'] else 'Bad only')
    btn_bad.button_style = ('warning' if nav['bad_only'] else '')
    pool = _filtered_idxs()
    if pool and target_idxs[nav['pos']] not in pool:
        nav['pos'] = target_idxs.index(pool[0])
    render_for(target_idxs[nav['pos']])


btn_prev = W.Button(description='◀ Prev', layout=W.Layout(width='auto'))
btn_next = W.Button(description='Next ▶', layout=W.Layout(width='auto'))
btn_bad  = W.Button(description='Bad only', layout=W.Layout(width='auto'))
btn_prev.on_click(lambda _: _step(-1))
btn_next.on_click(lambda _: _step(1))
btn_bad.on_click(_toggle_bad)

out = W.Output()
display(W.HBox([btn_prev, btn_next, btn_bad]))
display(out)

render_for(target_idxs[nav['pos']])


Output()

### 4c · Compare pipeline vs ground truth

For every calibrated frame in `data/manual_calibration/{slug}_*.json`, runs
PnLCalib + refine once on the source frame and overlays the result against the
ground truth:

- **white** lines: current pipeline output
- **cyan** lines: manual ground truth
- **red dots**: clicked landmarks

Reports mean pixel error per frame at 7 reference world points (center spot,
halfway-line endpoints, PA outer corners). Lower is better. Comparison images
land in `output/demo/diag/gt_compare/`.

In [8]:
%matplotlib inline
from src.manual_calibration import calibration_dir, load_calibration

GT_DIR = Path('../output/demo/diag/gt_compare')
GT_DIR.mkdir(parents=True, exist_ok=True)

seed_paths = sorted(calibration_dir().glob(f'{MATCH}_frame_*.json'))
print(f'Found {len(seed_paths)} calibrated frame(s) for {MATCH}')

cap_cmp = cv2.VideoCapture(str(Config.MATCH_VIDEOS[MATCH]))
fw_cmp  = int(cap_cmp.get(cv2.CAP_PROP_FRAME_WIDTH))
fhv_cmp = int(cap_cmp.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Reference world points used to score pipeline-vs-GT pixel deviation.
TEST_WORLD_PTS = [
    (52.5, 34.0,  0.0),
    (52.5,  0.0,  0.0), (52.5, 68.0,  0.0),
    (16.5, 13.84, 0.0), (16.5, 54.16, 0.0),
    (88.5, 13.84, 0.0), (88.5, 54.16, 0.0),
]


def _project(P, w):
    v = P @ np.array([w[0] - 52.5, w[1] - 34.0, w[2], 1])
    return None if v[2] <= 0 else (v[0] / v[2], v[1] / v[2])


errors = []
for sp in seed_paths:
    cal = load_calibration(sp)
    if cal.P_pnlcalib_convention is None:
        continue
    P_gt = np.array(cal.P_pnlcalib_convention, dtype=np.float64)
    abs_f = cal.frame_number

    cap_cmp.set(cv2.CAP_PROP_POS_FRAMES, abs_f)
    ok, frm = cap_cmp.read()
    if not ok:
        continue

    # Per-frame pipeline answer — no flow / smoother, just PnLCalib + refine.
    P_pred, status = predict_one_frame(frm, pnl_models, fw_cmp, fhv_cmp)
    if P_pred is not None:
        P_pred = refine_projection_to_lines(P_pred, frm)

    deltas = []
    if P_pred is not None:
        for w in TEST_WORLD_PTS:
            a = _project(P_pred, w)
            b = _project(P_gt, w)
            if a and b:
                deltas.append(float(np.hypot(a[0] - b[0], a[1] - b[1])))
    pix_err = float(np.mean(deltas)) if deltas else float('nan')
    errors.append((abs_f, pix_err, status, P_pred is not None))

    vis = frm.copy()
    if P_pred is not None:
        project_lines(vis, P_pred, color=(255, 255, 255), thickness=2)
    project_lines(vis, P_gt, color=(255, 200, 0), thickness=2)
    for _name, _pit, (px, py) in cal.correspondences:
        cv2.circle(vis, (int(px), int(py)), 6, (0, 0, 255), -1)
        cv2.circle(vis, (int(px), int(py)), 6, (255, 255, 255), 1)
    cv2.putText(vis, f'frame={abs_f}  pix_err={pix_err:.1f}  {status}',
                (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
    cv2.imwrite(str(GT_DIR / f'cmp_{abs_f:07d}.jpg'), vis)

cap_cmp.release()

print()
print(f'{"frame":>9} {"px_err":>8} {"ok":>3}  status')
for f, e, s, ok in sorted(errors):
    print(f'{f:>9d} {e:>8.1f} {"Y" if ok else "N":>3}  {s}')

valid = [e for _f, e, _s, ok in errors if ok and not np.isnan(e)]
if valid:
    print(f'\nMean pixel error across {len(valid)} GT frames: {np.mean(valid):.1f} px')
print(f'Wrote comparison images to {GT_DIR}')


Found 57 calibrated frame(s) for sut-mla

    frame   px_err  ok  status
    97200     11.0   Y  pnlcalib
    97210     14.6   Y  pnlcalib
    97220     36.0   Y  pnlcalib
    97230     11.9   Y  pnlcalib
    97240     90.4   Y  pnlcalib
    97250     34.4   Y  pnlcalib
    97260     12.8   Y  pnlcalib
    97270     12.5   Y  pnlcalib
    97280      8.5   Y  pnlcalib
    97290      6.8   Y  pnlcalib
    97300     30.9   Y  pnlcalib
    97310     29.5   Y  pnlcalib
    97320     25.8   Y  pnlcalib
    97330     10.4   Y  pnlcalib
    97340     21.5   Y  pnlcalib
    97350      2.5   Y  pnlcalib
    97360     10.1   Y  pnlcalib
    97370    162.5   Y  pnlcalib
    97380    146.3   Y  pnlcalib
    97390     40.1   Y  pnlcalib
    97400     68.3   Y  pnlcalib
    97410     19.9   Y  pnlcalib
    97420     20.3   Y  pnlcalib
    97430     28.8   Y  pnlcalib
    97440     46.0   Y  pnlcalib
    97450      0.0   Y  pnlcalib
    97460      0.0   Y  pnlcalib
    97470      0.0   Y  pnlcalib
   

### 4d · Label ball positions

YOLO loses the ball during dribbles and slow rolls. To make the demo's ball
overlay rock-solid, click the ball every Nth frame in the rendered window;
the renderer will linearly interpolate between your clicks and use those
positions in preference to YOLO. Annotations save to
`data/ball_annotations/{slug}.json` immediately on each click — the cell is
incremental, so close it and resume any time.

**How to use:**

1. Click on the ball in the displayed frame.
2. The widget auto-advances by `step` frames.
3. **Skip** if the ball isn't visible. **Delete** removes the current frame's
   annotation. **Save** is explicit but not needed (every click auto-saves).
4. The renderer uses your annotations within their range; it falls back to
   YOLO for frames outside, or for gaps wider than ~15 frames between
   neighbouring labels.

In [9]:
%matplotlib widget
import importlib, src.ball_annotation as BA; importlib.reload(BA)

BALL_LABEL_STEP = 5   # label one out of every N frames in the render window

_cap = cv2.VideoCapture(str(Config.MATCH_VIDEOS[MATCH]))
_fps = _cap.get(cv2.CAP_PROP_FPS) or 25.0
_cap.release()
_start_abs = int(D.parse_ts(START_TS) * _fps)
_end_abs   = _start_abs + int(DURATION_SEC * _fps) - 1

BA.build_ball_label_widget(
    MATCH,
    Config.MATCH_VIDEOS[MATCH],
    start_frame=_start_abs,
    end_frame=_end_abs,
    step=BALL_LABEL_STEP,
)

## 5 · Render (or re-render) the demo MP4

Each frame goes through:

- YOLO detection + BoT-SORT tracking (`persist=True`)
- Wide-shot gate — homography is skipped on close-ups so the flow tracker
  doesn't re-seed on a useless view
- PnLCalib → optical-flow propagation (gated on `check_projection_sanity` +
  `check_line_alignment`) → manual seed fallback → stale-hold from the smoother
- `refine_projection_to_lines` snaps the candidate P onto painted white pixels
- `ProjectionSmoother` EMA-blends across frames
- Per-player foot is projected to pitch coords for the speed/distance tracker
  (which accumulates through pre-roll so stats are warm by render frame 0)
- Render: white pitch lines, team-coloured ellipses + name badges, ball triangle,
  minimap, video-timestamp

Output: `output/demo/{slug}_{ts}_demo.mp4`. The summary at the bottom prints
homography source attribution, ball detection vs extrapolation count, and refine
stats so you can see at a glance what's working.

In [10]:
# ── Setup ───────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(str(Config.MATCH_VIDEOS[MATCH]))
fps = cap.get(cv2.CAP_PROP_FPS)
fw  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fhv = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

start_abs      = int(D.parse_ts(START_TS) * fps)
preroll_frames = int(PREROLL_SEC * fps)
render_frames  = int(DURATION_SEC * fps)
clip_start     = max(0, start_abs - preroll_frames)

ts_safe  = START_TS.replace(':', '-')
out_path = str(Config.OUTPUT_DIR / 'demo' / f'{MATCH}_{ts_safe}_demo.mp4')
Path(out_path).parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (fw, fhv))

# Reset BoT-SORT counter so track IDs are reproducible run-to-run.
if hasattr(yolo, 'predictor') and yolo.predictor is not None:
    yolo.predictor = None
from ultralytics.trackers.basetrack import BaseTrack
BaseTrack._count = 0

# Homography mode: pre-build a Gaussian-smoothed trajectory through the
# manual seeds and use that as the SOLE source of P within the seed range.
# PnLCalib / flow / refine / ProjectionSmoother all sit out within seeds;
# they only run before the first seed and after the last (where there is no
# GT track to interpolate). This kills the snap-and-drift artefact: every
# in-range frame is a smooth linear-interp blended through a Gaussian, no
# pipeline jitter mixed in.
#
# SEED_SIGMA controls Gaussian sigma in frames:
#   4 = tight (good when seeds are every 5-10 frames AND well-calibrated)
#   6 = balanced default
#   8 = wider; pulls in distant seeds across long gaps (>15 frames). Use
#       this when there are sparse zones — the surrounding dense seeds will
#       dominate over an outlier or a single bracketing seed across a fast
#       camera pan, instead of letting a 19-frame linear interp run wild.
SEED_SIGMA = 4.0

manual_seeds = load_match_seeds(MATCH)
sorted_seed_frames = sorted(manual_seeds.keys())
print(f'Manual seeds for {MATCH}: {len(manual_seeds)}  (smoothed seed track, sigma={SEED_SIGMA:.1f})')


def _build_smoothed_seed_track(seeds, sigma):
    """Linearly interpolate a dense P track between seeds, then Gaussian-smooth
    along time. Returns {abs_frame: P (3x4)} for every frame in the seed range.
    Frames outside the range fall back to the pipeline at render time."""
    if not seeds:
        return {}
    sf = sorted(seeds.keys())
    first, last = sf[0], sf[-1]
    n = last - first + 1
    if n < 2:
        return {first: seeds[first].copy()}
    track = np.empty((n, 3, 4), dtype=np.float64)
    after_idx = 1
    for offset in range(n):
        af = first + offset
        while after_idx < len(sf) and sf[after_idx] < af:
            after_idx += 1
        if af in seeds:
            track[offset] = seeds[af]
            continue
        f1, f2 = sf[after_idx - 1], sf[after_idx]
        t = (af - f1) / (f2 - f1)
        track[offset] = (1 - t) * seeds[f1] + t * seeds[f2]
    try:
        from scipy.ndimage import gaussian_filter1d
        smoothed = gaussian_filter1d(track, sigma=sigma, axis=0, mode='nearest')
    except ImportError:
        radius = int(np.ceil(3 * sigma))
        kx = np.arange(-radius, radius + 1)
        kernel = np.exp(-0.5 * (kx / sigma) ** 2); kernel /= kernel.sum()
        padded = np.pad(track, ((radius, radius), (0, 0), (0, 0)), mode='edge')
        smoothed = np.empty_like(track)
        for offset in range(n):
            window = padded[offset:offset + 2 * radius + 1]
            smoothed[offset] = (window * kernel[:, None, None]).sum(axis=0)
    return {first + i: smoothed[i] for i in range(n)}


smoothed_seed_track = _build_smoothed_seed_track(manual_seeds, SEED_SIGMA)

# Manual ball annotations (every-N-frame clicks). Render uses these
# in preference to YOLO + velocity extrapolation when available.
from src.ball_annotation import load_ball_annotations as _load_ball_anns, interpolate_ball as _interp_ball
ball_annotations = _load_ball_anns(MATCH)
if ball_annotations:
    _bf = sorted(ball_annotations.keys())
    print(f'Ball annotations for {MATCH}: {len(ball_annotations)} (range {_bf[0]}–{_bf[-1]})')
else:
    print(f'No ball annotations for {MATCH} yet (renderer will use YOLO + interp)')

# Pipeline state, used only for the (typically short) frames OUTSIDE the seed
# range — i.e. before the first seed and after the last. Inside the range the
# smoothed track does everything and these never tick.
smoother       = ProjectionSmoother(alpha=0.3, max_stale_frames=int(fps * 3))
motion_tracker = CameraMotionTracker()

# Per-player speed / distance
player_stats = D.PlayerStats(fps)

# Ball history (last 2 frames) for velocity-based extrapolation across gaps
_ball_bboxes, _ball_frames, _ball_pitch = [], [], []

# Stats
homog_stats = {k: 0 for k in (
    'smooth_seed', 'pnlcalib', 'flow', 'flow_rejected', 'manual_seed',
    'stale', 'none',
    'close_up', 'inference_fail', 'sanity_fail', 'line_fail',
)}
ball_hits = 0
ball_carry = 0
for _k in list(REFINE_STATS):
    REFINE_STATS[_k] = 0


# ── Render loop ─────────────────────────────────────────────────────────────
cap.set(cv2.CAP_PROP_POS_FRAMES, clip_start)

for frame_i in tqdm(range(preroll_frames + render_frames), desc='Render'):
    ret, frame = cap.read()
    if not ret:
        break

    is_render  = frame_i >= preroll_frames
    abs_frame  = clip_start + frame_i
    render_sec = (frame_i - preroll_frames) / fps

    # Detect + track
    boxes = yolo.track(
        frame, imgsz=1280,
        conf=min(Config.PLAYER_CONF_THRESHOLD, BALL_CONF_THRESHOLD),
        tracker=f'{Config.TRACKER_TYPE}.yaml',
        persist=True, verbose=False,
    )[0].boxes

    players = []
    best_ball_bbox, best_ball_conf = None, -1
    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i])
        bbox   = boxes.xyxy[i].cpu().numpy().astype(int).tolist()
        conf   = float(boxes.conf[i])
        tid    = int(boxes.id[i]) if boxes.id is not None else -1
        if cls_id == 0 and conf >= Config.PLAYER_CONF_THRESHOLD:
            players.append({'bbox': bbox, 'conf': conf, 'track_id': tid})
        elif cls_id == 1 and conf >= BALL_CONF_THRESHOLD and conf > best_ball_conf:
            best_ball_bbox, best_ball_conf = bbox, conf

    wide = D.is_wide_shot(players, fhv)

    # Homography. Within the manual-seed range we use the pre-built smoothed
    # seed track — PnLCalib/flow/refine/smoother all sit out so their jitter
    # cannot leak into the seeded portion. Outside the range (before first
    # seed / after last) the full pipeline runs.
    P_smooth = None
    if wide:
        seed_P = smoothed_seed_track.get(abs_frame)
        if seed_P is not None:
            P_smooth = seed_P
            homog_stats['smooth_seed'] = homog_stats.get('smooth_seed', 0) + 1
        else:
            P_raw, status = predict_one_frame(
                frame, pnl_models, fw, fhv,
                manual_seeds=manual_seeds, abs_frame=abs_frame,
            )
            tag = status
            if P_raw is not None:
                motion_tracker.seed(frame, P_raw)
            else:
                P_flow = motion_tracker.propagate(frame)
                if P_flow is not None:
                    player_boxes = [p['bbox'] for p in players]
                    if (check_projection_sanity(P_flow, player_boxes)
                            and check_line_alignment(P_flow, frame, min_score=0.12)):
                        P_raw, tag = P_flow, 'flow'
                    else:
                        motion_tracker.reset()
                        tag = 'flow_rejected'
            if P_raw is not None:
                P_raw = refine_projection_to_lines(P_raw, frame)
            P_smooth = smoother.update(P_raw)
            if P_smooth is not None and P_raw is None:
                tag = 'stale'
            elif P_smooth is None:
                tag = 'none'
            homog_stats[tag] = homog_stats.get(tag, 0) + 1
    else:
        motion_tracker.reset()
        # Drop stale per-player positions so speed doesn't teleport
        # when the camera returns to a wide shot.
        for p in players:
            if p['track_id'] >= 0:
                player_stats.drop(p['track_id'])
        homog_stats['close_up'] += 1

    # Pitch projection + speed/distance update
    player_pitch_pts: dict[int, tuple[np.ndarray, int]] = {}
    if wide and P_smooth is not None:
        for p in players:
            tid = p['track_id']
            if tid < 0:
                continue
            xy = D.project_foot(p['bbox'], P_smooth)
            if xy is None:
                continue
            display_tid = tid
            for thresh, remap in sorted(TRACK_REMAPS, key=lambda x: x[0]):
                if render_sec >= thresh:
                    display_tid = remap.get(display_tid, display_tid)
            info = PLAYER_NAMES.get(display_tid, (None, 2, None))
            name_for_dist, team_id, _ = info
            # Distance keys on player NAME so it accumulates across track-id
            # changes when the same player exits and re-enters the frame.
            canonical = name_for_dist if name_for_dist else f'track_{display_tid}'
            player_stats.update(tid, xy, canonical_id=canonical)
            player_pitch_pts[display_tid] = (xy, team_id)

    # Ball: update history, then choose what bbox to draw (detection or extrapolation)
    if best_ball_bbox is not None:
        ball_hits += 1
        _ball_bboxes.append(best_ball_bbox)
        _ball_frames.append(abs_frame)
        pitch_xy = D.project_foot(best_ball_bbox, P_smooth) if P_smooth is not None else None
        _ball_pitch.append(pitch_xy)
        if len(_ball_bboxes) > 2:
            _ball_bboxes.pop(0); _ball_frames.pop(0); _ball_pitch.pop(0)

    draw_bbox = None
    ball_pitch_pt = None

    # Ball position priority:
    #   1. Manual annotation (interpolated linearly between bracketing
    #      labels). When the user has labelled this clip's ball every N
    #      frames this wins — continuous and ground-truthed.
    #   2. YOLO detection this frame.
    #   3. Velocity-extrapolated YOLO from the last two detections.
    _ann_pos = _interp_ball(ball_annotations, abs_frame)
    if _ann_pos is not None:
        _ax, _ay = int(_ann_pos[0]), int(_ann_pos[1])
        _half = 8
        draw_bbox = [
            int(np.clip(_ax - _half, 0, fw - 1)),
            int(np.clip(_ay - _half, 0, fhv - 1)),
            int(np.clip(_ax + _half, 0, fw - 1)),
            int(np.clip(_ay + _half, 0, fhv - 1)),
        ]
        if P_smooth is not None:
            ball_pitch_pt = D.project_foot(draw_bbox, P_smooth)
    elif best_ball_bbox is not None:
        draw_bbox = best_ball_bbox
        ball_pitch_pt = _ball_pitch[-1] if _ball_pitch else None
    elif _ball_frames and (abs_frame - _ball_frames[-1]) <= BALL_MAX_GAP:
        gap  = abs_frame - _ball_frames[-1]
        last = _ball_bboxes[-1]
        if len(_ball_bboxes) == 2:
            dt = _ball_frames[-1] - _ball_frames[-2]
            if dt > 0:
                damp = 1.0 - gap / (BALL_MAX_GAP + 1)
                vx = (last[0] - _ball_bboxes[-2][0]) / dt * damp
                vy = (last[1] - _ball_bboxes[-2][1]) / dt * damp
                dx, dy = int(vx * gap), int(vy * gap)
                draw_bbox = [
                    int(np.clip(last[0] + dx, 0, fw - 1)),
                    int(np.clip(last[1] + dy, 0, fhv - 1)),
                    int(np.clip(last[2] + dx, 0, fw - 1)),
                    int(np.clip(last[3] + dy, 0, fhv - 1)),
                ]
                if P_smooth is not None:
                    ball_pitch_pt = D.project_foot(draw_bbox, P_smooth)
        else:
            draw_bbox = last
            ball_pitch_pt = _ball_pitch[-1] if _ball_pitch else None
        if is_render and draw_bbox is not None:
            ball_carry += 1

    if not is_render:
        continue

    # Render this frame
    out_frame = frame.copy()
    if wide and P_smooth is not None:
        project_lines(out_frame, P_smooth, color=(255, 255, 255), thickness=2)

    for p in players:
        raw_tid = p['track_id']
        display_tid = raw_tid
        for thresh, remap in sorted(TRACK_REMAPS, key=lambda x: x[0]):
            if render_sec >= thresh:
                display_tid = remap.get(display_tid, display_tid)
        info = PLAYER_NAMES.get(display_tid)
        name      = info[0] if info else None
        team_id   = info[1] if info else 2
        jersey_no = info[2] if info else None
        # Speed is per raw track (resets on new BoT-SORT id, which is fine —
        # there's no continuous motion across a gap). Distance is per-player
        # canonical id so re-entries continue accumulating.
        canonical = name if name else f'track_{display_tid}'
        D.draw_player_annotation(
            out_frame, p['bbox'], name, team_id,
            player_stats.speed(raw_tid) if wide else 0.0,
            player_stats.dist(canonical),
            display_tid, jersey_no,
        )

    if draw_bbox is not None:
        D.draw_ball_triangle(out_frame, draw_bbox)

    if wide and P_smooth is not None and player_pitch_pts:
        viewport = D.compute_viewport_polygon(P_smooth, fw, fhv)
        D.draw_minimap(out_frame, player_pitch_pts, ball_pitch_pt,
                       viewport_pts=viewport)

    D.draw_timestamp(out_frame, abs_frame, fps)
    writer.write(out_frame)

cap.release()
writer.release()


# ── Summary ─────────────────────────────────────────────────────────────────
total = sum(homog_stats.values())
print(f'\nHomography coverage ({total} frames):')
for k, v in sorted(homog_stats.items(), key=lambda x: -x[1]):
    if v:
        print(f'  {k:15s}: {v:4d} ({100*v/max(total,1):5.1f}%)')
print(f'\nBall: {ball_hits} detections + {ball_carry} extrapolated')
print(f"Refine: {REFINE_STATS['applied']} applied / {REFINE_STATS['attempts']} attempts"
      f" ({REFINE_STATS['too_few']} too-few-inliers, {REFINE_STATS['rejected_worse']} rejected-worse)")
print(f'Saved → {out_path}')


Manual seeds for sut-mla: 57  (smoothed seed track, sigma=4.0)
Ball annotations for sut-mla: 90 (range 97200–97645)


Render:   0%|          | 0/825 [00:00<?, ?it/s]


Homography coverage (825 frames):
  smooth_seed    :  431 ( 52.2%)
  pnlcalib       :  365 ( 44.2%)
  close_up       :   19 (  2.3%)
  manual_seed    :   10 (  1.2%)

Ball: 305 detections + 0 extrapolated
Refine: 362 applied / 375 attempts (0 too-few-inliers, 13 rejected-worse)
Saved → C:\Users\PC\Desktop\GitHub\football-computer-vision\output\demo\sut-mla_1-04-48_demo.mp4
